In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from tavily import TavilyClient
from typing import Any, Dict
############################################################
client = MultiServerMCPClient(
    {
        "kiwi-com-flight-search": {
            "transport": "sse",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

#######################################################
tavilyclient= TavilyClient()

@tool
def web_search_for_venue(query: str) ->Dict[str, Any]:
    """Search the web for wedding venues and locations."""
    return tavilyclient.search(query)

#########################################################

@tool
def generate_playlist(genre: str) ->str:
    """Search the music database and generate a playlist based on the genre."""

    playlists = {
        "jazz": "1. Take Five - Dave Brubeck\n2. So What - Miles Davis\n3. Feeling Good - Nina Simone",
        "pop": "1. Uptown Funk - Bruno Mars\n2. Shake It Off - Taylor Swift\n3. Blinding Lights - The Weeknd",
        "classical": "1. Clair de Lune - Debussy\n2. Four Seasons - Vivaldi\n3. Symphony No. 5 - Beethoven"
    }
    genre_lower = genre.lower()
    if genre_lower in playlists:
        return f"Here is a great {genre} playlist:\n{playlists[genre_lower]}"
    else:
        return f"Here is a custom {genre} playlist:\n1. Top {genre} Hit\n2. Classic {genre} Anthem\n3. Best of {genre}"    


In [ ]:
from langchain.agents import create_agent

flight_agent= create_agent(
    model="gpt-5-nano",
    tools= tools,
    system_prompt="You are a flight booking expert. Use your tools to find flights."
)

venue_agent= create_agent(
    model="gpt-5-nano",
    tools=[web_search_for_venue],
    system_prompt="You are a wedding venue expert. Use your tools to search the web for venues."
)

dj_agent= create_agent(
    model="gpt-5-nano",
    tools=[generate_playlist],
    system_prompt="You are a professional DJ. Use your tools to generate playlists based on genres."
)



In [16]:
from langchain.messages import HumanMessage

@tool
async def call_flight_agent(query: str) ->str:
    """Call the flight agent to handle flight and travel bookings."""
    response= await flight_agent.ainvoke(
        {
            "messages": [HumanMessage(content=query)]
        }
    )
    return response["messages"][-1].content


@tool
async def call_venue_agent(query: str) ->str:
    """Call the venue agent to find wedding venues and locations."""
    response= await venue_agent.ainvoke(
        {
            "messages": [HumanMessage(content=query)]
        }
    )
    return response["messages"][-1].content


@tool
async def call_dj_agent(query: str) ->str:
    """Call the DJ agent to create music playlists."""
    response=await dj_agent.ainvoke(
        {
            "messages": [HumanMessage(content=query)]
        }
    )
    return response["messages"][-1].content

In [17]:
from langgraph.checkpoint.memory import InMemorySaver
main_agent= create_agent(
    model="gpt-5-nano",
    tools=[call_flight_agent, call_venue_agent, call_dj_agent],
    system_prompt="""
        You are the main "Wedding and Travel Coordinator. 
        Listen to the user's request and call the appropriate subagents (flights, venue, DJ) to fulfill it. 
        Combine their answers into a beautiful final response.""",
    checkpointer=InMemorySaver()    
    )

In [18]:
from pprint import pprint
question = (
    "I'm planning a post-graduation wedding celebration! "
    "First, I need a flight from Egypt to the Maldives for July 15th. "
    "Second, I need you to find a beautiful beachfront wedding venue there. "
    "Finally, please generate a lively pop music playlist for the reception party."
)

response=  await main_agent.ainvoke(
    {
        "messages": [HumanMessage(content=question)]
    },
    {
        "configurable":{
            "thread_id":"2"
        }
    }

)

pprint(response)

TavilyKeylessLimitError: You reached the hourly keyless Tavily limit. To continue immediately, pay via x402 agentic payment (https://docs.tavily.com/documentation/machine-payments/x402) or sign up at https://tavily.com for a Tavily API key. Otherwise, retry after the time given in the Retry-After response header.